# STAT 764 · Meeting 8 — Tuning inside the pipeline

**Thursday, October 1**

On Tuesday you turned a dial — the penalty α — by hand. Today a search turns it for
you, and hands you back a score. Every tuned model you will ever report arrives with that
score attached, and the search **chose the model by that score**.

It is not an academic worry. Kaggle scores every competition twice: a public
leaderboard you can watch, and a private one on held-back data that decides the prizes.
The second one exists because people tune against the first.

**Today's one idea: the number you chose with cannot be the number you report.**

1. **The best of several numbers** — what choosing does to a score, in the simplest case.
2. **Search harder, and score the search** — where the damage is large, and the fix.
3. **The tool you will use** — `GridSearchCV` inside the pipeline, scored honestly.

In [ ]:
# ── Setup. Run this cell first; it takes a couple of seconds. ──
import os
import pathlib
import sys

# Find the course helpers (course/stat764.py): in your clone if this notebook sits inside
# it, or wherever STAT764_REPO points. Failing both, fetch the one file from GitHub.
here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import warnings

import numpy as np
import pandas as pd
from stat764 import load                    # the course's data loader

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A 300-house sample often has no pool at all, so Pool_QC is entirely missing and the
# imputer would say so on every fit. The column carries nothing there.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

# ── The data ──
ames_all = load("ames.csv")                 # all 2,930 Ames house sales
# Set aside the five houses over 4,000 square feet (see the note below this cell).
ames = ames_all[ames_all["Gr_Liv_Area"] <= 4000].reset_index(drop=True)
y = ames["SalePrice"]                       # the target in every part today, in dollars

# ── The columns. Part 1 uses the small design; Part 3 and the studio use the wide one. ──
NUM = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]   # Meetings 5-6
CAT = ["Neighborhood", "Central_Air"]
WIDE_NUM = [c for c in ames.select_dtypes("number").columns            # Tuesday: every column
            if c not in ("Order", "PID", "SalePrice")]       # not the ids, not the answer
WIDE_CAT = ames.select_dtypes(exclude="number").columns.tolist()

# ── The grids a search chooses from. The search tries each value; you do not. ──
ALPHAS = np.logspace(-1, 4, 11)               # ridge: 0.1, 0.3, 1, 3, ... 10,000
LASSO_ALPHAS = [30, 100, 300, 1000, 3000]     # the lasso works on a different scale


def make_pipe(numeric, categorical, model=None):
    """The Meeting 2 skeleton. It does not change all semester.

    Everything learned from data (the medians, the means and standard deviations, the list
    of categories) is learned INSIDE the pipeline, so it is relearned from the training rows
    every time the pipeline is fit. When a search fits it in every fold, for every alpha,
    no fold's scaler ever sees the rows that fold is scored on.
    """
    return Pipeline([
        ("prep", ColumnTransformer([
            # numeric columns: fill each gap with the column's median, then standardize
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), numeric),
            # text columns: a gap becomes its own category, "Missing"; then one 0/1 column
            # per category (a category never seen in training gets all zeros, not an error)
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]),
             categorical),
        ])),
        ("model", model or LinearRegression()),     # OLS unless you pass Ridge(), Lasso(), ...
    ])


# ── Helpers for Part 2. You do not need to read them to follow the meeting. ──
# Part 2 fits OLS over 100,000 times for its nested scores alone (20 splits x 200 models
# x 5 inner folds x 5 outer folds), so these skip the pipeline and solve least squares
# directly on a plain array with nothing missing. Same scores as cross_val_score with
# LinearRegression, nearly twenty times faster.
def cv_r2(X, y, folds):
    """Mean cross-validated R-squared of OLS — what cross_val_score reports, only faster."""
    scores = []
    for tr, te in folds:
        # fit OLS on the training rows: a column of 1s for the intercept, then least squares
        coef = np.linalg.lstsq(np.column_stack([np.ones(len(tr)), X[tr]]), y[tr], rcond=None)[0]
        # predict the rows held out of this fold, and score them: R-squared = 1 - SSE / SST
        pred = np.column_stack([np.ones(len(te)), X[te]]) @ coef
        scores.append(1 - ((y[te] - pred) ** 2).sum() / ((y[te] - y[te].mean()) ** 2).sum())
    return np.mean(scores)


def pick_best(X, y, candidates, folds):
    """The search: score every candidate set of columns by CV, keep the best."""
    scores = [cv_r2(X[:, cols], y, folds) for cols in candidates]   # one CV score each
    i = int(np.argmax(scores))                                      # the winner...
    return candidates[i], scores[i]     # ...and its own score: the best of all it tried


def nested_score(X, y, candidates, seed):
    """Nested CV: run the WHOLE search inside each outer fold, score its winner on that fold."""
    scores = []
    # 1. split the data into outer folds (shuffled differently from the inner folds)
    for tr, te in KFold(5, shuffle=True, random_state=seed + 1000).split(X):
        # 2. inside this outer training set, run the whole search: its own inner folds,
        #    every candidate. Its winner's score is thrown away (the `_`).
        inner = list(KFold(5, shuffle=True, random_state=seed).split(X[tr]))
        cols, _ = pick_best(X[tr], y[tr], candidates, inner)
        # 3. refit the winner on the outer training set; score it on the fold it never saw
        fit = LinearRegression().fit(X[tr][:, cols], y[tr])
        scores.append(r2_score(y[te], fit.predict(X[te][:, cols])))
    return np.mean(scores)              # the score of the SEARCH, not of any one model


print(f"  {len(ames):,} houses ({len(ames_all) - len(ames)} set aside)")

**One change from Tuesday: five houses are set aside.** They are the only ones over 4,000
square feet, and three of them are *partial sales* — enormous houses sold unfinished for a
fraction of their value (one is Meeting 5's \$160,000 house that a model priced at
\$1.5 million). Dean De Cock, who assembled this dataset, recommends removing them before
teaching with it. With them in, today's numbers mostly measure which fold those five
landed in.

**Today: what choosing does to a score.** Whenever you keep the best of several options
by its score — a random seed, a set of columns, a value of α — the score you kept comes
out too high; today we see that, measure how big it gets, and fix it.

## 1. The best of several numbers

Start with the simplest search there is: **one model, one dataset, and nothing chosen but
the split.** Fit the Meeting 5 pipeline on 200 different splits and look at the spread.
Then imagine someone who tries ten `random_state` values and reports the best.

**Before you run it:** how far above a typical split would "the best of ten" land?

In [ ]:
X = ames[NUM + CAT]

# The SAME pipeline on 200 different train/test splits. Only random_state changes, so
# any difference between these scores is the luck of which houses landed in the test set.
split_scores = []
for seed in range(200):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)
    model = make_pipe(NUM, CAT).fit(X_tr, y_tr)                  # learn from 75%...
    split_scores.append(r2_score(y_te, model.predict(X_te)))     # ...score on the other 25%
split_scores = np.array(split_scores)

typical = np.median(split_scores)       # what an honest single split usually reports

# "Try ten seeds, keep the best": cut the 200 scores into twenty groups of ten (twenty
# separate people, each trying ten seeds) and keep the largest in each group.
best_of_ten = split_scores.reshape(20, 10).max(axis=1)
print("  one pipeline, 200 different splits")
print(f"    typical (median)  R-squared {typical:.3f}")
print(f"    worst                       {split_scores.min():.3f}")
print(f"    best                        {split_scores.max():.3f}")
print("\n  try ten seeds and keep the best — done twenty separate times")
print(f"    best of ten, typically      {np.median(best_of_ten):.3f}   "
      f"({np.median(best_of_ten) - typical:+.3f} over typical)")
print(f"    it beat the typical split   {(best_of_ten > typical).sum()} of 20 times")

### What this shows

**Keeping the best of ten beat the typical split every single time — and all of it is
luck.** The model never changed. The data never changed. Only the draw did.

**Why:** each split's score is the model's true performance *plus noise*. When you keep the
largest of ten noisy numbers, you are keeping the one whose noise happened to be most
positive. This has a name — the **winner's curse**: *whenever you choose by a noisy number,
the number you chose by is too high.*

**The statistical word is *biased*.** Not too high once, by bad luck — too high *on
average*, every time you do it: 20 of 20 here. An estimate whose errors lean one way is
**biased**, and this one is **biased upward**. When the bias comes from choosing, it has a
name of its own: **selection bias**.

**Nothing leaked.** No test row reached the model; no feature was built from the answer.
**The score is still biased upward.** A leak is one way to get a flattering number — not
the only way.

## 2. Search harder, and score the search

Now a search people really run when a deadline is close: **try lots of genuinely different
models and keep the best.** Here that is 200 random sets of eight columns, plain OLS on
each, 150 houses to learn from, each model scored by 5-fold CV.

Report the winner's own CV score, and you are reporting a best-of-200. The honest
alternative is **nested cross-validation**, which scores the *search* instead of the winner:

1. split the data into outer folds;
2. inside each outer training set, run the **whole** search — all 200 models, CV and all;
3. score the model it picks on the outer fold the search never saw.

We compare the winner's score, the nested score, and the truth: the winner's score on
732 houses nobody touched. Twenty different splits, with **one** model and with **200**.

**Before you run it:** which of the three will be too high?

In [ ]:
# Only the columns with no missing values, as a plain array: nothing to impute, so the
# fast OLS helpers in the setup cell can stand in for the pipeline.
COMPLETE = [c for c in WIDE_NUM if ames[c].notna().all()]    # 25 columns, nothing to impute
Xc, yc = ames[COMPLETE].to_numpy(), y.to_numpy()

rows = []
for seed in range(20):                                        # twenty different splits
    rng = np.random.default_rng(seed)
    X_tr, X_te, y_tr, y_te = train_test_split(Xc, yc, test_size=0.25, random_state=seed)
    X_tr, y_tr = X_tr[:150], y_tr[:150]                       # only 150 houses to learn from
    # the 200 candidate models: 200 random sets of 8 of the 25 columns
    candidates = [rng.choice(len(COMPLETE), 8, replace=False) for _ in range(200)]
    folds = list(KFold(5, shuffle=True, random_state=seed).split(X_tr))
    for k in (1, 200):                        # search the first candidate alone, then all 200
        # the search, and the winner's own CV score: the best of k
        cols, winner = pick_best(X_tr, y_tr, candidates[:k], folds)
        # the truth: refit the winner on the 150 houses, score it on 732 nobody touched
        held = r2_score(y_te, LinearRegression().fit(X_tr[:, cols], y_tr).predict(X_te[:, cols]))
        rows.append({"models tried": k, "winner's own CV": winner,
                     "nested CV": nested_score(X_tr, y_tr, candidates[:k], seed),
                     "truth (held out)": held})

# One row per split and search size. Summarize over the twenty splits: the median of each
# score, and how often the winner's score beat nested, and nested beat the truth.
results = pd.DataFrame(rows)
results["winner above nested"] = results["winner's own CV"] > results["nested CV"]
results["nested above truth"] = results["nested CV"] > results["truth (held out)"]
by_k = results.groupby("models tried")
table = by_k[["winner's own CV", "nested CV", "truth (held out)"]].median().round(3)
for col in ("winner above nested", "nested above truth"):
    table[col] = by_k[col].sum().astype(int).astype(str) + " of 20"
print("  medians over 20 splits")
print(table.to_string())

### What this shows

- **With one model, nothing was chosen** — the winner's score and the nested score agree,
  and which is higher is a coin flip.
- **With 200 models, the winner's own score is biased upward** — above the nested score in
  17 of 20 splits, and above the truth. That is the winner's curse — selection bias — now
  from a search.
- **The nested score lands on the truth**: above it about half the time (9 of 20), below
  it the other half. Its errors go both ways — that is what a (nearly) **unbiased**
  estimate looks like — and it used only the same 150 houses. No extra data.

**Why nested works:** the search is part of what you are evaluating. Nested CV runs it
fresh inside every outer fold, so each outer fold scores a model the search chose
*without seeing that fold* — exactly the situation of a brand-new house.

## 3. The tool you will use

You will not write that loop again. **`GridSearchCV` is itself an estimator**, so you can
hand it to `cross_val_score` — and the whole search then runs inside every outer fold.

Tuesday's setting: the wide design, 300 houses, ridge, eleven values of α. Two details:

- `"model__alpha"` means *the `alpha` of the step named `"model"`* — the search reaches
  **inside** the pipeline, so the imputer and the scaler are refit in every fold, for every
  α. Meeting 4's rule, applied to tuning.
- `best_score_` is the **winner's** score — the best of eleven.

In [ ]:
Xw = ames[WIDE_NUM + WIDE_CAT]
X_train, X_test, y_train, y_test = train_test_split(Xw, y, test_size=0.25, random_state=764)
X300, y300 = X_train.iloc[:300], y_train.iloc[:300]               # Tuesday's 300 houses
# X_test is the held-out set. It is used ONCE, in the last line of this cell.

# Two sets of folds, two jobs.
inner = KFold(5, shuffle=True, random_state=764)     # INNER folds: the search CHOOSES alpha
outer = KFold(5, shuffle=True, random_state=1764)    # OUTER folds: they GRADE the whole search

# The search, on the INNER folds: 5-fold CV of the whole pipeline at each of the eleven
# alphas. It keeps the alpha with the best mean score, then refits it on all 300 houses.
search = GridSearchCV(make_pipe(WIDE_NUM, WIDE_CAT, Ridge()), {"model__alpha": ALPHAS},
                      cv=inner, scoring="r2")
search.fit(X300, y300)

# Nested CV, on the OUTER folds: hand the SEARCH itself to cross_val_score. Each outer fold
# runs a fresh search — its own inner folds and all — on the other four outer folds, and
# grades the alpha it picks on the fold it never saw. (cross_val_score fits copies, so
# `search` above is left as it was.)
nested = cross_val_score(search, X300, y300, cv=outer, scoring="r2")

print(f"  the winner: alpha = {search.best_params_['model__alpha']:g}")
print(f"  best_score_   the winner's own CV score       {search.best_score_:.3f}")
print(f"  nested CV     the whole search, scored        {nested.mean():.3f}")
print(f"                its five outer-fold grades      {', '.join(f'{g:.3f}' for g in nested)}")
print(f"  held out      touched once, at the very end   {r2_score(y_test, search.predict(X_test)):.3f}")

In [ ]:
# Open the box: the same nested score as three plain loops — no GridSearchCV, no
# cross_val_score. Same `outer` and `inner` folds, same grid, same pipeline as above.
grades = []
for out_tr, out_te in outer.split(X300):                 # LOOP 1: each outer fold
    X_o, y_o = X300.iloc[out_tr], y300.iloc[out_tr]      # the 240 houses the search may use
    cv_score = {}
    for a in ALPHAS:                                     # LOOP 2: every alpha on the grid
        scores = []
        for in_tr, in_va in inner.split(X_o):            # LOOP 3: inner folds, 192 / 48
            fit = make_pipe(WIDE_NUM, WIDE_CAT, Ridge(alpha=a)).fit(X_o.iloc[in_tr],
                                                                    y_o.iloc[in_tr])
            scores.append(r2_score(y_o.iloc[in_va], fit.predict(X_o.iloc[in_va])))
        cv_score[a] = np.mean(scores)                    # this alpha's CV score
    best = max(cv_score, key=cv_score.get)               # the search picks the best alpha...
    final = make_pipe(WIDE_NUM, WIDE_CAT, Ridge(alpha=best)).fit(X_o, y_o)   # ...refits on 240
    grades.append(r2_score(y300.iloc[out_te], final.predict(X300.iloc[out_te])))
    print(f"  outer fold {len(grades)}: picked alpha {best:.3g}, "
          f"graded {grades[-1]:.3f} on {len(out_te)} houses it never saw")

print(f"\n  nested score, by hand            {np.mean(grades):.3f}")
print(f"  cross_val_score(search), above   {nested.mean():.3f}")

### What this shows

**The cell above opens the box.** Three loops: the outer folds *grade*; the α loop and
the inner folds *choose*. Written out by hand, it lands on exactly the one-liner's number.

**Here the three roughly agree.** Eleven values of α are eleven *nearly identical* models,
scored on the *same* folds: their scores rise and fall together, so the best of them is
barely above the rest. A small, smooth grid has little to be cursed by, so the upward bias
in `best_score_` is small. Two hundred genuinely different models had far more.

**So which number do you report?** The **nested** one — it scores the procedure you will
actually run, search included. `best_score_` happened to be close this time; you only
know that because you checked. And the held-out number is the one you get to look at
**once**, at the very end: look at it in order to choose, and it stops being held out.

## The whole procedure, start to finish

**Plenty of data — keep a test set.**

1. Split off a test set first. Do not look at it again until step 4 — not even to choose
   between models.
2. On the training set, run the search: `GridSearchCV` uses cross-validation to choose α.
3. The search refits the winner on the whole training set — automatically.
4. Score that model on the test set, **once**. That is the number you report.

**Little data, like 300 houses — nested CV.** A small test set would be a noisy grade:
the five outer folds above graded the same procedure on 60 houses each and got anywhere
from 0.821 to 0.932.

1. **Choose:** `search.fit(X, y)` on all the data picks α and refits it on all of it —
   automatically. That is the model you use.
2. **Report:** `cross_val_score(search, X, y)` — the nested score. That is the number you
   report for it.

**Either way:** everything learned from data — the imputer, the scaler, α — is learned
inside the pipeline and the search. **Choosing between models too? Make the model one
more thing the search chooses, and both recipes stay the same.** What not to do: tune
ridge, the lasso and a forest separately and report the best of their three scores —
that is choosing again, and the winner's number is biased upward.

Today's studio does both on purpose — 300 houses to work with, the rest held back as the
truth — so you can see the numbers side by side.

## Studio

Tuesday's 300 houses, the wide design. **All tuning inside resampling.**

**Finish this one.** Do Part 3 again for the **lasso**: `Lasso(max_iter=20000)` with
`{"model__alpha": LASSO_ALPHAS}`. Report the α it chose, `best_score_`, the nested
estimate, and — **once** — the held-out score. Which number would you put in a report,
and why?

**If you have time: peek on purpose.** Choose ridge's α by its score on the **test** set
(fit each α on `X300`, score each on `X_test`, keep the best) and report that score. How
much did peeking buy you — and could anyone reading your report tell that you peeked?

In [ ]:
# YOUR CODE HERE — Part 3's cell is the model; copy it and change the two pieces.
# Give it a new name: the extension still needs Part 3's `search`.
#
# lasso_search = GridSearchCV(make_pipe(WIDE_NUM, WIDE_CAT, Lasso(max_iter=20000)), ...)
#   1. fit the search on X300, y300        -> the alpha it chose, and best_score_
#   2. cross_val_score on the search       -> the nested estimate
#   3. score it on X_test, once, last      -> the held-out number

## Compare

1. For the lasso: how far apart were `best_score_` and the nested estimate? Which number
   goes in the report?
2. If you peeked: what did it buy — and is "not much" a reason to do it?

## Exit ticket

> A report says: *"after tuning, the model achieved a cross-validated R-squared of
> 0.91."* What one question would you ask before believing it — and what answer would
> satisfy you?

**Answer in the cell below — two or three sentences — then save this notebook and upload
it to Canvas under "Studio notebook — Day 8". Your two studio points need this
answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Cawley & Talbot (2010),** *On over-fitting in model selection and subsequent
selection bias in performance evaluation*, JMLR 11:2079–2107. The paper that measured
this properly — and showed the selection bias can be as large as the difference between
the methods being compared.

📗 **ESL §7.10.3,** *Does cross-validation really work?* — the optional reading, and the
honest answer to the question in its title.

📗 **`RidgeCV` and `LassoCV`.** Shortcuts that tune α internally. They are a search too:
their score is a winner's score, and the same rules apply.

🚫 **Reporting `best_score_` as the model's performance.** It is the score you chose by —
biased upward by the choosing.
On a small, smooth grid it may be close; you only know that by checking.